# 🗂️ Notebook 2: Code Deployment (CI/CD) — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/code-deployment
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Entities

- **Pipeline** — config (DAG of stages).
- **Run** — one execution of the pipeline.
- **Stage** — unit of work.
- **Artifact** — immutable output.
- **Deployment** — pinned artifact in an env.

## Pydantic models

We use `pydantic` for data validation — it forces us to think about types, required fields, and invariants up front.

In [ ]:
from datetime import datetime
from typing import Literal
from typing import Optional
from pydantic import BaseModel

class Stage(BaseModel):
    name: str
    cmd: str
    depends_on: list[str] = []

class Pipeline(BaseModel):
    repo: str
    stages: list[Stage]

class Run(BaseModel):
    id: int
    pipeline: str
    commit_sha: str
    status: Literal["queued","running","passed","failed"] = "queued"
    started_at: Optional[datetime] = None

class Deployment(BaseModel):
    env: Literal["dev","staging","prod"]
    artifact: str    # "repo@sha"
    strategy: Literal["rolling","canary","bluegreen"]

## HTTP APIs

| Method | Path | What |
|---|---|---|
| POST | `/hooks/push` | Git webhook entry |
| GET | `/runs/{id}` | Run status |
| POST | `/deployments` | Deploy a specific artifact |
| POST | `/deployments/{id}/rollback` | Roll back to previous |


## Quick demo

In [ ]:
# Topological order of stages (DAG)
from collections import defaultdict, deque

def topo(stages):
    graph = defaultdict(list); indeg = defaultdict(int)
    names = [s['name'] for s in stages]
    for n in names: indeg[n]         # ensure key exists
    for s in stages:
        for d in s['depends_on']:
            graph[d].append(s['name']); indeg[s['name']] += 1
    q = deque([n for n in names if indeg[n]==0])
    out = []
    while q:
        n = q.popleft(); out.append(n)
        for m in graph[n]:
            indeg[m] -= 1
            if indeg[m] == 0: q.append(m)
    return out

stages = [
    {"name":"build", "depends_on":[]},
    {"name":"unit",  "depends_on":["build"]},
    {"name":"int",   "depends_on":["build"]},
    {"name":"deploy","depends_on":["unit","int"]},
]
print(topo(stages))

## Takeaways

- Small, typed models make the service boundary crisp.
- Public APIs hide internal IDs and expose human-friendly resources.
- Write one happy-path test per endpoint before scaling out.